# Credit Card Fraud Analytics — Fraud Analysis

## Phase 4: Fraud Analysis

This phase focuses specifically on fraudulent transactions and translates exploratory findings into fraud-oriented analytical metrics.

Objectives:
- Profile fraudulent transactions
- Compare fraud and normal populations
- Analyze fraud concentration by amount and time
- Measure fraud exposure
- Identify statistically meaningful differences
- Build business-oriented fraud KPIs
- Prepare evidence for the dashboard and final report

Important:
- V1–V28 are anonymized PCA components.
- The dataset covers approximately two days, so time analysis is limited to short-period and hour-of-day patterns.
- No customer, merchant, location, or category-level conclusions are possible because those fields are not available.


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 2. Load Clean Dataset


In [ ]:
DATA_PATH = "../data/creditcard_clean.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Fraud transactions: {(df['Class'] == 1).sum():,}")
print(f"Normal transactions: {(df['Class'] == 0).sum():,}")


## 3. Create Fraud and Normal Samples


In [ ]:
fraud = df[df["Class"] == 1].copy()
normal = df[df["Class"] == 0].copy()

print(f"Fraud sample size: {len(fraud):,}")
print(f"Normal sample size: {len(normal):,}")


## 4. Fraud Transaction Profile


In [ ]:
fraud_profile = pd.DataFrame({
    "metric": [
        "Fraud transactions",
        "Fraud transaction rate (%)",
        "Fraud amount total",
        "Fraud amount mean",
        "Fraud amount median",
        "Fraud amount maximum",
        "Fraud zero-amount transactions",
        "Fraud zero-amount percentage (%)"
    ],
    "value": [
        len(fraud),
        df["Class"].mean() * 100,
        fraud["Amount"].sum(),
        fraud["Amount"].mean(),
        fraud["Amount"].median(),
        fraud["Amount"].max(),
        (fraud["Amount"] == 0).sum(),
        (fraud["Amount"] == 0).mean() * 100
    ]
})

display(fraud_profile)


## 5. Fraud Amount Distribution


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(fraud["Amount"], bins=50)
plt.title("Fraudulent Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Number of Fraud Transactions")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(np.log1p(fraud["Amount"]), bins=50)
plt.title("Log-Transformed Fraud Amount Distribution")
plt.xlabel("log(1 + Amount)")
plt.ylabel("Number of Fraud Transactions")
plt.tight_layout()
plt.show()


## 6. Fraud Amount Concentration


In [ ]:
fraud_amount_quantiles = fraud["Amount"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("Fraud Amount")

display(fraud_amount_quantiles)


## 7. Top Fraudulent Transactions by Amount


In [ ]:
top_fraud_transactions = (
    fraud[["Time", "Amount", "Class"]]
    .sort_values("Amount", ascending=False)
    .head(20)
)

display(top_fraud_transactions)


These records are descriptive examples only. A large transaction is not automatically more suspicious than a smaller one.


## 8. Fraud Amount Bands


In [ ]:
amount_bins = [-0.01, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, np.inf]
amount_labels = [
    "0–10", "10–25", "25–50", "50–100", "100–250",
    "250–500", "500–1,000", "1,000–2,500",
    "2,500–5,000", "5,000+"
]

fraud["AmountBand"] = pd.cut(
    fraud["Amount"],
    bins=amount_bins,
    labels=amount_labels
)

fraud_band_summary = (
    fraud.groupby("AmountBand", observed=False)
    .agg(
        fraud_transactions=("Class", "size"),
        fraud_amount=("Amount", "sum"),
        average_amount=("Amount", "mean"),
        median_amount=("Amount", "median")
    )
)

fraud_band_summary["transaction_share_pct"] = (
    fraud_band_summary["fraud_transactions"]
    / len(fraud) * 100
)

fraud_band_summary["amount_share_pct"] = (
    fraud_band_summary["fraud_amount"]
    / fraud["Amount"].sum() * 100
)

display(fraud_band_summary)


## 9. Fraud Amount Share by Band


In [ ]:
plot_data = fraud_band_summary.reset_index()

plt.figure(figsize=(12, 5))
plt.bar(
    plot_data["AmountBand"].astype(str),
    plot_data["amount_share_pct"]
)
plt.title("Share of Fraud Amount by Amount Band")
plt.xlabel("Amount Band")
plt.ylabel("Share of Total Fraud Amount (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Fraud Time Analysis


In [ ]:
fraud["TimeHours"] = fraud["Time"] / 3600
fraud["HourOfDay"] = (fraud["Time"] // 3600) % 24

fraud_hourly = (
    fraud.groupby("HourOfDay")
    .agg(
        fraud_transactions=("Class", "size"),
        fraud_amount=("Amount", "sum"),
        average_amount=("Amount", "mean")
    )
)

fraud_hourly["transaction_share_pct"] = (
    fraud_hourly["fraud_transactions"] / len(fraud) * 100
)

fraud_hourly["amount_share_pct"] = (
    fraud_hourly["fraud_amount"] / fraud["Amount"].sum() * 100
)

display(fraud_hourly)


## 11. Fraud Transactions by Hour


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(
    fraud_hourly.index,
    fraud_hourly["fraud_transactions"]
)
plt.title("Fraud Transactions by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Fraud Transactions")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


## 12. Fraud Amount by Hour


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(
    fraud_hourly.index,
    fraud_hourly["fraud_amount"]
)
plt.title("Fraud Amount by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Total Fraud Amount")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


## 13. Fraud Rate by Hour


In [ ]:
df["HourOfDay"] = (df["Time"] // 3600) % 24

hourly_risk = (
    df.groupby("HourOfDay")
    .agg(
        transactions=("Class", "size"),
        fraud_transactions=("Class", "sum")
    )
)

hourly_risk["fraud_rate_pct"] = (
    hourly_risk["fraud_transactions"]
    / hourly_risk["transactions"] * 100
)

display(hourly_risk)


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(
    hourly_risk.index,
    hourly_risk["fraud_rate_pct"]
)
plt.title("Fraud Rate by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


## 14. Fraud vs Normal Amount Comparison


In [ ]:
amount_comparison = pd.DataFrame({
    "metric": [
        "Mean",
        "Median",
        "25th percentile",
        "75th percentile",
        "90th percentile",
        "95th percentile",
        "99th percentile",
        "Maximum"
    ],
    "Normal": [
        normal["Amount"].mean(),
        normal["Amount"].median(),
        normal["Amount"].quantile(.25),
        normal["Amount"].quantile(.75),
        normal["Amount"].quantile(.90),
        normal["Amount"].quantile(.95),
        normal["Amount"].quantile(.99),
        normal["Amount"].max()
    ],
    "Fraud": [
        fraud["Amount"].mean(),
        fraud["Amount"].median(),
        fraud["Amount"].quantile(.25),
        fraud["Amount"].quantile(.75),
        fraud["Amount"].quantile(.90),
        fraud["Amount"].quantile(.95),
        fraud["Amount"].quantile(.99),
        fraud["Amount"].max()
    ]
})

display(amount_comparison)


## 15. Statistical Test — Transaction Amount


In [ ]:
# Mann-Whitney U is used because transaction amounts are highly skewed.
mann_whitney_result = stats.mannwhitneyu(
    normal["Amount"],
    fraud["Amount"],
    alternative="two-sided"
)

print(f"U statistic: {mann_whitney_result.statistic:,.0f}")
print(f"p-value: {mann_whitney_result.pvalue:.6g}")


### Interpretation

The Mann–Whitney U test evaluates whether the distributions of transaction amounts differ between normal and fraudulent transactions.

A small p-value indicates statistical evidence of a distributional difference. Statistical significance does not by itself indicate practical importance.


## 16. Effect Size — Rank-Biserial Correlation


In [ ]:
u = mann_whitney_result.statistic
n1 = len(normal)
n2 = len(fraud)

rank_biserial = 1 - (2 * u) / (n1 * n2)

effect_size_summary = pd.DataFrame({
    "metric": ["Mann-Whitney U", "p-value", "Rank-biserial effect size"],
    "value": [u, mann_whitney_result.pvalue, rank_biserial]
})

display(effect_size_summary)


Effect size provides additional context because a statistically significant result can occur even when the practical difference is small.


## 17. Fraud Feature Mean Differences


In [ ]:
v_features = [f"V{i}" for i in range(1, 29)]

feature_comparison = pd.DataFrame({
    "normal_mean": normal[v_features].mean(),
    "fraud_mean": fraud[v_features].mean()
})

feature_comparison["mean_difference"] = (
    feature_comparison["fraud_mean"]
    - feature_comparison["normal_mean"]
)

feature_comparison["absolute_mean_difference"] = (
    feature_comparison["mean_difference"].abs()
)

feature_comparison = feature_comparison.sort_values(
    "absolute_mean_difference",
    ascending=False
)

display(feature_comparison)


## 18. Top PCA Features by Mean Difference


In [ ]:
top_features = feature_comparison.head(10).sort_values(
    "mean_difference"
)

plt.figure(figsize=(9, 6))
plt.barh(
    top_features.index,
    top_features["mean_difference"]
)
plt.title("Top PCA Features by Fraud vs Normal Mean Difference")
plt.xlabel("Fraud Mean − Normal Mean")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


These are statistical differences in anonymized PCA features. They should not be interpreted as business variables.


## 19. Fraud Exposure KPIs


In [ ]:
total_amount = df["Amount"].sum()
fraud_amount = fraud["Amount"].sum()

fraud_exposure_kpis = pd.DataFrame({
    "KPI": [
        "Total transactions",
        "Fraud transactions",
        "Fraud transaction rate (%)",
        "Total transaction amount",
        "Fraud amount",
        "Fraud amount share (%)",
        "Average fraud amount",
        "Median fraud amount",
        "Maximum fraud amount"
    ],
    "Value": [
        len(df),
        len(fraud),
        len(fraud) / len(df) * 100,
        total_amount,
        fraud_amount,
        fraud_amount / total_amount * 100,
        fraud["Amount"].mean(),
        fraud["Amount"].median(),
        fraud["Amount"].max()
    ]
})

display(fraud_exposure_kpis)


## 20. Fraud Concentration — Top 10% of Fraud Transactions


In [ ]:
fraud_sorted = fraud.sort_values("Amount", ascending=False).reset_index(drop=True)

top_10_count = max(1, int(np.ceil(len(fraud_sorted) * 0.10)))

top_10_fraud_amount = fraud_sorted.head(top_10_count)["Amount"].sum()
top_10_fraud_share = top_10_fraud_amount / fraud_amount * 100

print(f"Top 10% fraud transaction count: {top_10_count:,}")
print(f"Top 10% fraud amount share: {top_10_fraud_share:.2f}%")


This metric measures whether fraud monetary exposure is concentrated in a small number of fraudulent transactions.


## 21. Fraud Concentration — Cumulative Curve


In [ ]:
fraud_sorted["cumulative_amount_share"] = (
    fraud_sorted["Amount"].cumsum() / fraud_amount * 100
)

fraud_sorted["cumulative_transaction_share"] = (
    (np.arange(len(fraud_sorted)) + 1) / len(fraud_sorted) * 100
)

plt.figure(figsize=(9, 6))
plt.plot(
    fraud_sorted["cumulative_transaction_share"],
    fraud_sorted["cumulative_amount_share"]
)
plt.title("Cumulative Fraud Amount Concentration")
plt.xlabel("Cumulative Fraud Transactions (%)")
plt.ylabel("Cumulative Fraud Amount (%)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 22. Zero-Amount Fraud Analysis


In [ ]:
zero_fraud = fraud[fraud["Amount"] == 0]
nonzero_fraud = fraud[fraud["Amount"] > 0]

zero_amount_fraud_summary = pd.DataFrame({
    "metric": [
        "Zero-amount fraud transactions",
        "Zero-amount fraud share (%)",
        "Non-zero fraud transactions",
        "Non-zero fraud share (%)"
    ],
    "value": [
        len(zero_fraud),
        len(zero_fraud) / len(fraud) * 100,
        len(nonzero_fraud),
        len(nonzero_fraud) / len(fraud) * 100
    ]
})

display(zero_amount_fraud_summary)


Zero-amount fraud records are retained as valid observations unless there is external evidence that they are invalid.


## 23. Business-Oriented Risk Segments


In [ ]:
df["AmountBand"] = pd.cut(
    df["Amount"],
    bins=amount_bins,
    labels=amount_labels
)

risk_segments = (
    df.groupby("AmountBand", observed=False)
    .agg(
        transactions=("Class", "size"),
        fraud_transactions=("Class", "sum"),
        total_amount=("Amount", "sum"),
        fraud_amount=("Amount", lambda x: x[df.loc[x.index, "Class"] == 1].sum())
    )
)

risk_segments["fraud_rate_pct"] = (
    risk_segments["fraud_transactions"]
    / risk_segments["transactions"] * 100
)

risk_segments["fraud_amount_share_pct"] = (
    risk_segments["fraud_amount"]
    / fraud_amount * 100
)

display(risk_segments)


## 24. Phase 4 KPI Table


In [ ]:
phase4_kpis = pd.DataFrame({
    "KPI": [
        "Fraud transaction rate (%)",
        "Fraud amount share (%)",
        "Average fraud amount",
        "Median fraud amount",
        "Maximum fraud amount",
        "Top 10% fraud amount concentration (%)",
        "Zero-amount fraud share (%)",
        "Amount Mann-Whitney p-value",
        "Amount rank-biserial effect size"
    ],
    "Value": [
        len(fraud) / len(df) * 100,
        fraud_amount / total_amount * 100,
        fraud["Amount"].mean(),
        fraud["Amount"].median(),
        fraud["Amount"].max(),
        top_10_fraud_share,
        len(zero_fraud) / len(fraud) * 100,
        mann_whitney_result.pvalue,
        rank_biserial
    ]
})

display(phase4_kpis)


## 25. Save Phase 4 Outputs


In [ ]:
OUTPUT_DIR = "../data"

fraud_profile.to_csv(f"{OUTPUT_DIR}/fraud_profile.csv", index=False)
fraud_band_summary.to_csv(f"{OUTPUT_DIR}/fraud_amount_band_summary.csv")
fraud_hourly.to_csv(f"{OUTPUT_DIR}/fraud_hourly_summary.csv")
hourly_risk.to_csv(f"{OUTPUT_DIR}/fraud_hourly_risk.csv")
amount_comparison.to_csv(f"{OUTPUT_DIR}/fraud_amount_comparison.csv", index=False)
effect_size_summary.to_csv(f"{OUTPUT_DIR}/fraud_amount_statistical_test.csv", index=False)
feature_comparison.to_csv(f"{OUTPUT_DIR}/fraud_feature_comparison.csv")
risk_segments.to_csv(f"{OUTPUT_DIR}/fraud_risk_segments.csv")
phase4_kpis.to_csv(f"{OUTPUT_DIR}/phase4_fraud_kpis.csv", index=False)

print("Phase 4 output tables saved successfully.")


## 26. Phase 4 Conclusions

After running the notebook, document evidence-based findings for:

1. Fraud transaction frequency and monetary exposure
2. Fraud amount distribution
3. Fraud concentration across amount bands
4. Fraud concentration across time of day
5. Statistical difference between normal and fraud transaction amounts
6. PCA features with the largest fraud-vs-normal mean differences
7. Concentration of monetary fraud exposure among the largest fraud transactions
8. Zero-amount fraud behavior

Avoid causal claims. These analyses identify associations and patterns in the observed dataset.

### Next Phase

**Phase 5 — Anomaly Detection**

We will use unsupervised techniques to identify unusual transactions and compare anomaly scores with the known fraud labels.
